# Phase 3: Generator Benchmarking

This notebook systematically trains and evaluates candidate synthetic data generators (CTGAN, TVAE, CopulaGAN, CTAB-GAN+, Gaussian Copula, WGAN-GP, TabDDPM) to determine the best model for generating 5 million transactions.

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, recall_score, precision_score
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

# Load the data
print("Loading real data...")
# Read a subset for benchmarking if necessary
transactions = pd.read_parquet('../../data/interim/transactions.parquet')
# Using a 50k sample to keep benchmark training times manageable
df = transactions.sample(n=min(50000, len(transactions)), random_state=42)
print(f"Data shape: {df.shape}")
print(f"Fraud rate: {df['is_fraud'].mean():.4f}")


## 2. Data Preprocessing (Base Columns Only)

In [ ]:
# Define base columns (excluding derived ones like log_amount, amount_local_npr etc.)
# Based on the EDA guidelines.
target = 'is_fraud'

# Example base columns (adjust according to the actual schema)
base_columns = [
    'transaction_date', 'transaction_time', 'sender_account_id', 'receiver_account_id',
    'transaction_type', 'amount_npr', 'original_currency', 'channel', 'merchant_category',
    'device_type', 'is_fraud'
]

# Keep only columns that exist
available_base_cols = [c for c in base_columns if c in df.columns]
df_base = df[available_base_cols]

# Train/Test Split (80/20) for TSTR evaluation
train_data, test_data = train_test_split(df_base, test_size=0.2, random_state=42, stratify=df_base[target])
print(f"Training set: {train_data.shape}, Test set: {test_data.shape}")


## 3. TRTR Baseline (Train Real, Test Real)

In [ ]:
# Prepare data for XGBoost (encoding categoricals)
def prepare_for_xgb(df_data, target_col):
    X = df_data.drop(columns=[target_col])
    y = df_data[target_col]
    
    # Simple label encoding for categoricals for the baseline
    for col in X.select_dtypes(include=['object', 'category']).columns:
        X[col] = X[col].astype('category')
        
    return X, y

X_train_real, y_train_real = prepare_for_xgb(train_data, target)
X_test_real, y_test_real = prepare_for_xgb(test_data, target)

# Train baseline
print("Training TRTR XGBoost Baseline...")
xgb_baseline = XGBClassifier(random_state=42, enable_categorical=True, use_label_encoder=False, eval_metric='logloss')
xgb_baseline.fit(X_train_real, y_train_real)

# Evaluate
preds = xgb_baseline.predict(X_test_real)
probs = xgb_baseline.predict_proba(X_test_real)[:, 1]

trtr_auc = roc_auc_score(y_test_real, probs)
trtr_prauc = average_precision_score(y_test_real, probs)
trtr_recall = recall_score(y_test_real, preds)

print(f"TRTR AUC-ROC: {trtr_auc:.4f}")
print(f"TRTR PR-AUC: {trtr_prauc:.4f}")
print(f"TRTR Fraud Recall: {trtr_recall:.4f}")


## 4. Generator Training & Sampling

In [ ]:
import sys
sys.path.append('../../')
# Import generators
# Note: Ensure these generators are fully implemented and can be instantiated.
from src.generation.gaussian_copula_generator import GaussianCopulaGenerator
from src.generation.ctgan_generator import CTGANGenerator
from src.generation.tvae_generator import TVAEGenerator
from src.generation.copulagan_generator import CopulaGANGenerator
# from src.generation.ctabganplus_generator import CTABGANPlusGenerator
# from src.generation.wgan_gp_generator import WGANGPGenerator
# from src.generation.tabddpm_generator import TabDDPMGenerator

# Dictionary of generators to evaluate
generators = {
    'Gaussian Copula': GaussianCopulaGenerator,
    'CTGAN': CTGANGenerator,
    'TVAE': TVAEGenerator,
    'CopulaGAN': CopulaGANGenerator,
    # Add others as they are verified to be fully operational
}

synthetic_datasets = {}
training_times = {}

for name, GenClass in generators.items():
    print(f"\n--- Training {name} ---")
    start_time = time.time()
    
    # Initialize and train
    gen = GenClass() 
    # Try/except block to handle missing implementations gracefully
    try:
        gen.fit(train_data)
        
        # Generate synthetic data of same size
        syn_data = gen.generate(len(train_data))
        
        elapsed = time.time() - start_time
        synthetic_datasets[name] = syn_data
        training_times[name] = elapsed
        print(f"Finished in {elapsed:.2f} seconds.")
    except Exception as e:
        print(f"Failed to train {name}: {e}")


## 5. Statistical Fidelity Evaluation

In [ ]:
from scipy.stats import ks_2samp

fidelity_results = {}
numeric_cols = train_data.select_dtypes(include=[np.number]).columns.drop(target, errors='ignore')

for name, syn_data in synthetic_datasets.items():
    print(f"\nEvaluating Fidelity: {name}")
    ks_scores = []
    
    for col in numeric_cols:
        stat, pval = ks_2samp(train_data[col].dropna(), syn_data[col].dropna())
        ks_scores.append(stat)
        
    avg_ks = np.mean(ks_scores)
    fidelity_results[name] = {'Avg KS (lower is better)': avg_ks}
    print(f"Average KS Statistic: {avg_ks:.4f}")


## 6. ML Utility Evaluation (TSTR)

In [ ]:
utility_results = {}

for name, syn_data in synthetic_datasets.items():
    print(f"\nEvaluating TSTR: {name}")
    
    X_train_syn, y_train_syn = prepare_for_xgb(syn_data, target)
    
    # Handle potentially unseen categories in test data by aligning categorical types
    for col in X_train_syn.select_dtypes(include=['category']).columns:
        combined = set(X_train_syn[col].cat.categories).union(set(X_test_real[col].cat.categories))
        X_train_syn[col] = X_train_syn[col].cat.set_categories(combined)
        X_test_real_aligned = X_test_real.copy()
        X_test_real_aligned[col] = X_test_real_aligned[col].cat.set_categories(combined)
    
    try:
        xgb_syn = XGBClassifier(random_state=42, enable_categorical=True, use_label_encoder=False, eval_metric='logloss')
        xgb_syn.fit(X_train_syn, y_train_syn)
        
        preds_syn = xgb_syn.predict(X_test_real_aligned)
        probs_syn = xgb_syn.predict_proba(X_test_real_aligned)[:, 1]
        
        auc = roc_auc_score(y_test_real, probs_syn)
        prauc = average_precision_score(y_test_real, probs_syn)
        recall = recall_score(y_test_real, preds_syn)
        
        utility_results[name] = {
            'TSTR AUC': auc,
            'TSTR PR-AUC': prauc,
            'TSTR Recall': recall
        }
        print(f"AUC: {auc:.4f}, PR-AUC: {prauc:.4f}, Recall: {recall:.4f}")
    except Exception as e:
        print(f"Failed TSTR for {name}: {e}")


## 7. Privacy Evaluation (Distance to Closest Record)

In [ ]:
# Note: DCR computation is O(N^2) and very slow. 
# We'll compute it on a small sample of 1000 records for the benchmark.
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
import pandas as pd

def compute_dcr(real, synthetic, sample_size=1000):
    real_sample = real.sample(n=min(sample_size, len(real)), random_state=42)
    syn_sample = synthetic.sample(n=min(sample_size, len(synthetic)), random_state=42)
    
    # Use only numeric columns for DCR for simplicity in this baseline
    num_cols = real_sample.select_dtypes(include=[np.number]).columns
    
    if len(num_cols) == 0:
        return np.nan
        
    scaler = StandardScaler()
    real_scaled = scaler.fit_transform(real_sample[num_cols].fillna(0))
    syn_scaled = scaler.transform(syn_sample[num_cols].fillna(0))
    
    nn = NearestNeighbors(n_neighbors=1, algorithm='ball_tree').fit(real_scaled)
    distances, _ = nn.kneighbors(syn_scaled)
    
    return np.mean(distances)

privacy_results = {}

for name, syn_data in synthetic_datasets.items():
    print(f"\nEvaluating Privacy (DCR): {name}")
    try:
        avg_dcr = compute_dcr(train_data, syn_data)
        privacy_results[name] = {'Avg DCR (higher is better)': avg_dcr}
        print(f"Average DCR: {avg_dcr:.4f}")
    except Exception as e:
        print(f"Failed DCR for {name}: {e}")


## 8. Final Scoring Matrix

In [ ]:
# Compile all results into a single DataFrame
final_scores = pd.DataFrame(index=synthetic_datasets.keys())

# Add TRTR baseline as a row for reference
baseline_row = pd.DataFrame({
    'Training Time (s)': [0],
    'Avg KS (lower is better)': [0],
    'TSTR AUC': [trtr_auc],
    'TSTR PR-AUC': [trtr_prauc],
    'TSTR Recall': [trtr_recall],
    'Avg DCR (higher is better)': [np.nan]
}, index=['TRTR Baseline (Real Data)'])


# Populate the generator metrics
for name in synthetic_datasets.keys():
    final_scores.loc[name, 'Training Time (s)'] = training_times.get(name, np.nan)
    if name in fidelity_results:
        final_scores.loc[name, 'Avg KS (lower is better)'] = fidelity_results[name].get('Avg KS (lower is better)')
    if name in utility_results:
        final_scores.loc[name, 'TSTR AUC'] = utility_results[name].get('TSTR AUC')
        final_scores.loc[name, 'TSTR PR-AUC'] = utility_results[name].get('TSTR PR-AUC')
        final_scores.loc[name, 'TSTR Recall'] = utility_results[name].get('TSTR Recall')
    if name in privacy_results:
        final_scores.loc[name, 'Avg DCR (higher is better)'] = privacy_results[name].get('Avg DCR (higher is better)')

# Append baseline and display
final_scores = pd.concat([baseline_row, final_scores])
display(final_scores.round(4))

# Highlight the best generator based on TSTR Recall (most important for fraud)
if not final_scores.drop('TRTR Baseline (Real Data)').empty:
    best_gen = final_scores.drop('TRTR Baseline (Real Data)')['TSTR Recall'].idxmax()
    print(f"\n=== Conclusion ===")
    print(f"Based on the TSTR Recall metric, the recommended generator is: **{best_gen}**")
